# Experiment with plotly maps 

Goals: 
1. use our custom tiles layers
2. add selected nhd lines from athena with different symbology

In [ ]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import shapely.geometry
from shapely import wkb

from util import prepare_gdf_for_athena
from util.athena import aoi_query_to_local_parquet
from util.figures import get_aoi_outline_trace, get_zoom_and_center
from util.pandas import load_gdf_from_pq


In [ ]:
# input and output paths
geojson_file = Path(r"C:\nardata\localcode\rs-reports-gen\src\reports\rpt_pbr_explorer\example\east_montana_portion.geojson")
aoi_gdf = gpd.read_file(geojson_file)
local_path = Path(r"C:\nardata\pydataroot\reportnotebook\various_maps")
parquet_path = local_path / "parquet"

In [ ]:
# get stream data for input from Athena, place in parquet_path

stream_sql_template = """SELECT fcode, gnis_name, reachcode, flowdir, ftype, nhdplusid, streamleve, streamorde, geom_wkb
FROM input_geom, ext_rpt.us_usgs_nhdplushr_network_flowline 
WHERE {prefilter_condition} AND {intersects_condition}
"""

query_gdf, simplification_results = prepare_gdf_for_athena(aoi_gdf)
aoi_query_to_local_parquet(stream_sql_template, geometry_field_expression="ST_GeomFromBinary(geom_wkb)", geom_bbox_field="geometry_bbox", aoi_gdf=query_gdf, local_path=parquet_path, intersection_measure="length")


In [ ]:
# load from local parquet
parquet_path = local_path / "parquet"
df = load_gdf_from_pq(parquet_path, geometry_col='geom_wkb')
print(df)

In [ ]:
# biggest streams streamleve 3 and streamorde 10
# streamleve starts at 1 at coast; stream order 1 starts at headwaters, means no tributaries
big_streams = df.query("streamleve==3 and streamorde ==10")
print(big_streams)

In [ ]:
# make a map of Yellowstone River
geo_df = df.query("gnis_name=='Yellowstone River'")
print(geo_df.shape)
print(geo_df.columns)

# -----------------------------
# tweak options (easy to change)
# -----------------------------
MAP_STYLE = "open-street-map"
MAP_HEIGHT = 650
MAP_MARGIN = {"l": 8, "r": 8, "t": 8, "b": 8}
SHOW_LEGEND = True

STREAM_LINE_WIDTH = 2
STREAM_LINE_COLOR = "#1f77b4"
STREAM_OPACITY = 0.9

AOI_LINE_WIDTH = 4
AOI_LINE_COLOR = "#ff7f0e"
AOI_NAME = "AOI"

VECTOR_TILE_SOURCE = "https://tiles.riverscapes.net/pmTiles/huc4/1.0/{z}/{x}/{y}.pbf"
VECTOR_TILE_SOURCE_LAYER = "geometry"
VECTOR_TILE_LINE_COLOR = "#2b8cbe"
VECTOR_TILE_LINE_WIDTH = 2

# Build AOI trace once, then add it to the line_map figure.
aoi_trace = get_aoi_outline_trace(aoi_gdf)

# Build line_map arrays; append None to create a break between separate linestrings.
lats = []
lons = []
names = []

for feature, name in zip(geo_df.geometry, geo_df.gnis_name):
    if isinstance(feature, shapely.geometry.linestring.LineString):
        linestrings = [feature]
    elif isinstance(feature, shapely.geometry.multilinestring.MultiLineString):
        linestrings = feature.geoms
    else:
        continue

    for linestring in linestrings:
        x, y = linestring.xy
        lats.extend(y)
        lons.extend(x)
        names.extend([name] * len(y))

        # None prevents Plotly from drawing connector lines between segments.
        lats.append(None)
        lons.append(None)
        names.append(None)

zoom, center = get_zoom_and_center(geo_df, "geom_wkb")

# Base stream map
fig = px.line_map(
    lat=lats,
    lon=lons,
    hover_name=names,
    map_style=MAP_STYLE,
    zoom=int(zoom),
    center=center,
    title="Yellowstone River + AOI",
)

# Style stream trace(s) created by px.line_map.
fig.update_traces(
    line={"width": STREAM_LINE_WIDTH, "color": STREAM_LINE_COLOR},
    opacity=STREAM_OPACITY,
    name="NHD flowlines",
)

# Overlay AOI trace and style it for visibility.
aoi_trace.name = AOI_NAME
if hasattr(aoi_trace, "line") and aoi_trace.line is not None:
    aoi_trace.line.width = AOI_LINE_WIDTH
    aoi_trace.line.color = AOI_LINE_COLOR
fig.add_trace(aoi_trace)

# Optional custom vector tile layer under traces.
fig.update_layout(
    map_layers=[
        {
            "below": "traces",
            "sourcetype": "vector",
            "source": [VECTOR_TILE_SOURCE],
            "sourcelayer": VECTOR_TILE_SOURCE_LAYER,
            "type": "line",
            "color": VECTOR_TILE_LINE_COLOR,
            "line": {"width": VECTOR_TILE_LINE_WIDTH},
        }
    ]
)

# Layout polish
fig.update_layout(
    margin=MAP_MARGIN,
    height=MAP_HEIGHT,
    autosize=True,
    showlegend=SHOW_LEGEND,
)

fig.show(config={"responsive": True})


In [ ]:
# get real z/x/y from known lat/lon and zoom
import math
lat = center.get('lat')
lon=center.get('lon')
z=int(zoom)
n=2*z
x=int((lon+180.0)/360.0 * n)
y=int((1.0-math.log(math.tan(math.radians(lat))+1.0/math.cos(math.radians(lat)))/math.pi)/2.0*n)
print({'z':z,'x':x,'y':y})

In [ ]:
# fetch tile for this
import requests
import mapbox_vector_tile as mvt
url = f"https://tiles.riverscapes.net/pmTiles/huc4/1.0/{z}/{x}/{y}.pbf" # 204 http - no results
url_that_works = "https://tiles.riverscapes.net/pmTiles/huc4/1.0/5/5/11.pbf" # both mvt and pbf extensions work
print(url)
r=requests.get(url_that_works,timeout=20)
print('status',r.status_code,'bytes',len(r.content),'content-type',r.headers.get('content-type'))
d=mvt.decode(r.content);
print('layers',list(d.keys()))  # 'geometry' is the only layer
